In [ ]:
from miditok import REMI, TokenizerConfig
from pathlib import Path
import numpy as np
import keras
import tensorflow as tf

### Define model name, tokenizer name, and timesteps

In [ ]:
model_name = "model.keras"
tokenizer_name = "tokenizer.json"
timesteps = 50

### Setup dataset

In [ ]:
from helpers import create_random_midi_folder

# Create a random dataset under the folders midis_train, midis_test, midis_val
create_random_midi_folder([(10, "midis_train"), (2, "midis_test"), (2, "midis_val")])

### Create a Tokenizer

In [ ]:
tokenizer = REMI(TokenizerConfig(use_programs=True))
paths_midis = list(Path("midis_train").glob('**/*.mid'))
paths_midis.extend(list(Path("midis_test").glob('**/*.mid')))
paths_midis.extend(list(Path("midis_val").glob('**/*.mid')))

tokenizer.train(
    vocab_size=2000,
    model="BPE",
    files_paths=paths_midis,
)
tokenizer.save(tokenizer_name)

### Load a tokenizer

In [ ]:
tokenizer = REMI(params=tokenizer_name)
paths_midis = list(Path("midis_train").glob('**/*.mid'))
paths_midis.extend(list(Path("midis_test").glob('**/*.mid')))
paths_midis.extend(list(Path("midis_val").glob('**/*.mid')))

### Setup training data

In [14]:
from helpers import load_songs_random, setup_timestep_data

paths_train = list(Path("midis_train").glob('**/*.mid'))
paths_val = list(Path("midis_val").glob('**/*.mid'))

# Load in randomly selected songs
train_data = load_songs_random(tokenizer, paths_train, 10)
val_data = load_songs_random(tokenizer, paths_val, 1)

vocab_size = tokenizer.vocab_size

x_train, y_train = setup_timestep_data(train_data, timesteps)
x_val, y_val = setup_timestep_data(val_data, timesteps)

### Create a model

In [ ]:
dim_emb = 256              # Dimensionality of the embedding space
num_heads = 8              # Number of attention heads
dim_ff = 512               # Dimensionality of the feed-forward network
input_length = timesteps

inputs = keras.Input(shape=(input_length,))

embedding = keras.layers.Embedding(input_dim=vocab_size, output_dim=dim_emb)(inputs)

positions = tf.range(start=0, limit=input_length, delta=1)
position_embeddings = keras.layers.Embedding(input_dim=input_length, output_dim=dim_emb)(positions)

# Sum the embeddings and positional encodings
x = embedding + position_embeddings

# Multi-head attention layer
attn_output = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim_emb)(x, x)
attn_output = keras.layers.Dropout(0.1)(attn_output)
out1 = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

# Feed-forward network
ffn_output = keras.layers.Dense(dim_ff, activation='relu')(out1)
ffn_output = keras.layers.Dense(dim_emb)(ffn_output)
ffn_output = keras.layers.Dropout(0.1)(ffn_output)
out2 = keras.layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

# Flatten the output and add a final Dense layer
out_flat = keras.layers.Flatten()(out2)
outputs = keras.layers.Dense(vocab_size, activation='softmax')(out_flat)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy')


In [ ]:
# Fit the model
model.fit(x_train, y_train, epochs=30, batch_size=64)

In [ ]:
model.save(model_name)

In [ ]:
model = keras.models.load_model(model_name)

In [ ]:
generated_notes = []
x = tokenizer.encode("midis_train/Shepherd, Arthur, Piano Sonata No.1, Op.4, MMJyYi2Tz0E.mid")
seed = np.array(x[0:timesteps])

num_notes_to_generate = 500

for _ in range(num_notes_to_generate):
    seed = np.reshape(seed, (1, len(seed)))
    seed = tf.convert_to_tensor(seed, dtype=tf.int32)
    prediction = np.argmax(model.predict(seed, verbose=0), axis=1)

    seed = np.append(seed.numpy(), prediction)[-timesteps:]
    
    generated_notes.append(int(prediction[0]))
    

print(generated_notes)

In [ ]:
tokenizer(generated_notes).dump_midi("generated.mid")

### Export this model to the expected file format of the GUI

In [ ]:
from helpers import create_model_folder

output_directory_name = "transformer_model"

create_model_folder(output_directory_name, model_name, tokenizer_name, timesteps, True)